# BSCP sack-train-ml — Train a Run

## ▶ To start: click **Runtime → Run all** (or press `Cmd/Ctrl + F9`)

This notebook trains a YOLO model for a registry run id and streams metrics + final artifacts back to Supabase.

**Before you Run all, check:**
1. **GPU runtime is set** — Runtime → Change runtime type → T4 (free) or better.
2. **You have a Supabase service-role key** — the notebook will prompt for it.
3. **Run id is in the URL** as `?run_id=...`. If not, paste it manually when prompted.
4. **The setup cell prints a git SHA** after syncing `main`; the training log should show the same SHA before export starts.

## 1. Resolve the run id from the URL

In [ ]:
from urllib.parse import urlparse, parse_qs

RUN_ID = ''
try:
    from google.colab import _message  # type: ignore
    raw = _message.blocking_request('get_url', timeout_sec=5)
    # Newer Colab returns {'url': '...'}; older returns the URL string directly.
    url = raw.get('url', '') if isinstance(raw, dict) else (raw or '')
    qs = parse_qs(urlparse(url).query)
    RUN_ID = qs.get('run_id', [''])[0]
except Exception as exc:
    print('Could not auto-read run_id from the URL:', exc)

print('Run id:', RUN_ID or '(none — paste below)')


In [ ]:
if not RUN_ID:
    RUN_ID = input('Paste run id: ').strip()
assert RUN_ID, 'run_id is required'
import os
os.environ['BSCP_RUN_ID'] = RUN_ID

In [ ]:
# GPU sanity check — fail fast if Runtime → Change runtime type wasn't set to a GPU.
import subprocess, sys
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
    if not out:
        raise RuntimeError('nvidia-smi reported no GPU')
    print('GPU detected:', out)
except FileNotFoundError:
    raise SystemExit(
        '\n[GPU required]\n'
        'Runtime → Change runtime type → Hardware accelerator → T4 GPU (or better), then Run all.'
    )
except Exception as e:
    raise SystemExit(f'\n[GPU check failed] {e}\nSwitch the runtime to GPU and re-run.')


## 2. Install dependencies + sync the repo

In [ ]:
# Deliberately NOT `pip install ultralytics` — an unpinned resolve is what let a
# five-hour-old broken release into a run. The version comes from the repo's
# pyproject, installed by the editable install in the repo-sync cell below.
%pip install --quiet onnx onnxsim pyyaml numpy 2>&1 | tail -3


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/pitikorn-pam/sack-train-ml.git'
REPO_DIR = Path('/content/sack-train-ml')
BRANCH = 'main'

if REPO_DIR.exists():
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

GIT_SHA = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True).strip()
os.environ['BSCP_GIT_SHA'] = GIT_SHA
print('Repo synced at', GIT_SHA)

# Editable install — show stderr if anything goes wrong
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('--- pip stdout ---')
    print(result.stdout[-2000:])
    print('--- pip stderr ---')
    print(result.stderr[-2000:])
    raise SystemExit(f'pip install -e failed (exit {result.returncode})')

# Belt-and-braces: also add src/ to sys.path so the import works even if the
# editable .pth file isn't picked up by this kernel session.
src_path = str(REPO_DIR / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Smoke-test the import right here so failures surface in this cell, not the next.
import importlib
if 'sack_train_ml' in sys.modules:
    importlib.reload(sys.modules['sack_train_ml'])
import sack_train_ml  # noqa: F401

print('sack_train_ml installed + importable')


## 3. Supabase auth (paste service-role key)

In [ ]:
import os
from getpass import getpass

# Public Supabase project URL — safe to hardcode.
SUPABASE_URL = 'https://eccwaoouvusinuvuybsr.supabase.co'

SUPABASE_SERVICE_ROLE_KEY = getpass('SUPABASE_SERVICE_ROLE_KEY: ')
assert SUPABASE_SERVICE_ROLE_KEY.startswith('eyJ'), 'service_role_key looks malformed (should be a JWT)'

os.environ['SUPABASE_URL'] = SUPABASE_URL
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = SUPABASE_SERVICE_ROLE_KEY
print('Auth env set.')

## 4. Connectivity check

In [ ]:
import os
from sack_train_ml.supabase_client import RegistryClient

client = RegistryClient()
run = client.fetch_run(os.environ['BSCP_RUN_ID'])
print('Run status:', run['status'])
print('Config keys:', list((run.get('config_yaml') or {}).keys()))

client.log_step(os.environ['BSCP_RUN_ID'], 1, 'init', 'info',
                f"colab notebook attached · git={os.environ.get('BSCP_GIT_SHA', '?')}")

## 5. (optional) train from a dataset already on this runtime

By default the pipeline downloads the dataset named in the run config. Set
`DATASET_SOURCE` below to a folder (containing `data.yaml`) or a `.zip` of one to
use that instead — handy for a locally-cleaned export. It sets `BSCP_DATASET_DIR`,
which `train_for_run.py` picks up as `--dataset-dir`.


In [ ]:
# Leave DATASET_SOURCE = '' to use the dataset from the run config (default).
# Otherwise: a dataset FOLDER holding data.yaml, or a .zip of one.
#   Drive folder : '/content/drive/MyDrive/SACK_DATASET_JULY_SEG_11'
#   uploaded zip : '/content/SACK_DATASET_JULY_SEG_11.zip'
DATASET_SOURCE = ''

import os, zipfile
from pathlib import Path

os.environ.pop('BSCP_DATASET_DIR', None)

def _find_root(d: Path) -> Path:
    """Return the folder holding data.yaml (descend one level if the zip nested it)."""
    if (d / 'data.yaml').exists():
        return d
    subs = [p for p in d.iterdir() if p.is_dir() and (p / 'data.yaml').exists()]
    if len(subs) == 1:
        return subs[0]
    raise FileNotFoundError(f'no data.yaml in {d} or a single subfolder of it')

if not DATASET_SOURCE:
    print('DATASET_SOURCE empty → using the dataset from the run config.')
else:
    src = Path(DATASET_SOURCE).expanduser()
    assert src.exists(), f'not found: {src}'

    if src.suffix.lower() == '.zip':
        dest = Path('/content/datasets') / src.stem
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src) as zf:
            zf.extractall(dest)
        print(f'extracted {src.name} → {dest}')
        root = _find_root(dest)
    else:
        root = _find_root(src)

    n_train = len(list((root / 'train' / 'images').glob('*.jpg')))
    assert n_train > 0, f'{root}/train/images has no .jpg'
    for split in ('train', 'valid', 'test'):
        imgs = len(list((root / split / 'images').glob('*.jpg')))
        lbls = len(list((root / split / 'labels').glob('*.txt')))
        print(f'  {split:6s} imgs={imgs:6d} labels={lbls}')

    os.environ['BSCP_DATASET_DIR'] = str(root)
    print('\nBSCP_DATASET_DIR =', root)
    print('train_for_run.py will use this and skip the bundle download.')


### (optional) resume an interrupted run

Point `RESUME_FROM` at the `last.pt` of a run that was killed mid-training. The
checkpoint carries the optimizer, EMA and epoch counter, so training continues
from the next epoch — epoch count and hyperparameters come from the checkpoint,
not from the run config.

A run that finished all its epochs stores `epoch: -1` and **cannot** be resumed;
the cell says so instead of failing later.


In [ ]:
# '' = train from config.source_weights (default). Otherwise a path to last.pt.
RESUME_FROM = ''

import os
from pathlib import Path

os.environ.pop('BSCP_RESUME_FROM', None)

if not RESUME_FROM:
    print('RESUME_FROM empty → fresh training run.')
else:
    ckpt = Path(RESUME_FROM).expanduser()
    assert ckpt.is_file(), f'not found: {ckpt}'

    import torch
    ck = torch.load(ckpt, map_location='cpu', weights_only=False)
    done = ck.get('epoch', -1)
    total = (ck.get('train_args') or {}).get('epochs')
    print(f'checkpoint: epoch={done}  train_args.epochs={total}')

    if done is None or done < 0:
        raise SystemExit(
            'This checkpoint finished its schedule (epoch=-1) — nothing to resume.\n'
            'Use it as source_weights for a new run instead.'
        )
    if total is not None and done + 1 >= total:
        raise SystemExit(f'Already at epoch {done + 1} of {total} — nothing to resume.')

    print(f'→ will resume at epoch {done + 2} of {total}')
    os.environ['BSCP_RESUME_FROM'] = str(ckpt.resolve())
    print('BSCP_RESUME_FROM =', os.environ['BSCP_RESUME_FROM'])


## 6. Run the training pipeline

Calls `scripts/train_for_run.py`, which orchestrates every stage and streams metrics live to the dashboard via the `training-callback` edge function.

**HEF compile (step 6b)** runs *in-flow* when the run was created with **Compile HEF** enabled in the web form (`config.compile_options.compile_hef`). It compiles an INT8 `.hef` in this same Colab session via a dedicated DFC virtualenv (the gated DFC wheel is pulled from R2 by key, calib images are sampled from the dataset), then uploads `.hef` + `.hef.meta.yaml` into the same version. It is **failure-safe**: if the compile fails, `.pt`/`.onnx` are still published.

Set `SKIP_HEF=1` to force-skip the compile (e.g. a quick re-run) even if it was enabled.

In [ ]:
import os
skip = '--skip-hef' if os.environ.get('SKIP_HEF', '').lower() in ('1', 'true', 'yes') else ''
!python -u /content/sack-train-ml/scripts/train_for_run.py --run-id $BSCP_RUN_ID $skip

## 7. Done

The run row is now `succeeded` (or `failed` with an error log entry). Artifacts are uploaded to R2 and the `versions` row is created. The web dashboard will reflect status live via Supabase Realtime.